# ColabのL4 GPUでvLLMを立てる

ColabのL4 GPU上で、RAGアプリ（`rag_web` と `ingest/`）が使うvLLMを起動し、手元の開発機から OpenAI 互換API（`/v1/chat/completions`・`/v1/embeddings` など）を叩けるようにする。起動する組は、セル0の `VLLM_PROFILE` で選ぶ。

| `VLLM_PROFILE` | 起動するモデル | 用途 |
|---|---|---|
| `"chat+embed"`（既定） | `openai/gpt-oss-20b`（`gpt-oss-20b`）と `BAAI/bge-m3`（`bge-m3`） | 回答の生成と、検索用の埋め込み |
| `"vlm"` | `Qwen/Qwen3-VL-8B-Instruct`（`qwen3-vl-8b`） | 取り込み時の画像説明 |

括弧内は `--served-model-name`（RAG側の `VLLM_*_MODEL` と同じ名前）。VLMは単独でGPUの大半を使うため、チャット・埋め込みとは同居させず、組を切り替えて使う。

**手順**
1. ランタイム → ランタイムのタイプを変更 → GPU（L4）を選択してから、上から順にセルを実行する
2. ngrokの無料アカウントを作り、ダッシュボードでauthtokenを控えておく（未取得なら ngrok.com で登録できる）。Colabのシークレット `NGROK_AUTHTOKEN` を使う場合、ノートブックをアップロードし直したときは「ノートブックからのアクセス」を入れ直すこと（入れないと入力に切り替わる）
3. セル9が表示する `VLLM_*` の行を、開発機の `.env` に書き写す（ノートブックはファイルに書かない）。手元から試すときは末尾の「手元から試す」を使う

**構成**: ngrok → 認証プロキシ（8000番、`Authorization: Bearer <キー>` を検証）→ vLLM（chat 8001番・embed 8002番・vlm 8003番）。プロキシはパスの先頭（`/chat`・`/embed`・`/vlm`）で転送先を選び、先頭を外して `/v1/...` を渡す。そのため、ngrokのURLは1つで済む。

```
VLLM_CHAT_URL=https://xxxx.ngrok-free.app/chat
VLLM_EMBED_URL=https://xxxx.ngrok-free.app/embed
```

URLは `/v1` で終わらせない（RAG側が `/v1/...` を足す）。

**注意**
- Colabのランタイムはアイドルや時間経過で切断される。常時稼働のAPIではなく、使う時だけ起動するものとして扱うこと。
- ngrokのURLは誰でも到達できるが、`Authorization: Bearer` のキーが一致しないリクエストはプロキシが401で拒否する。旧方式の `X-API-Key` ヘッダーは受け付けない。それでもURLとAPIキーは他人に共有しないこと。
- 初回はvLLMの導入（数分）とモデルのダウンロード（`chat+embed` で約15GB、`vlm` で約17GB）で時間がかかる。
- GPUメモリの割合（セル0）は実機で未確認の初期値である。起動時に CUDA out of memory で落ちたら、セル0のコメントに沿って調整する。


In [ ]:
# 0. 設定。後続のセルはここの値を使う（値を変えたらこのセルから実行し直す）。
import os

VLLM_VENV = "/content/vllm-env"
# 起動する組。
#   "chat+embed"（既定）: gpt-oss-20b と bge-m3 を1枚のL4に同居させる
#   "vlm"               : Qwen3-VL-8B だけを立てる
# VLMはbf16で約17GBあり、チャットと同居できないため別の組にしてある。
VLLM_PROFILE = "chat+embed"
PROXY_PORT = 8000

# 組ごとに起動するサービス。ポートはRAGアプリの既定（8001/8002/8003）に合わせ、
# プロキシ（8000番）がパスの先頭（route）で振り分ける。served_name は RAG側の
# VLLM_*_MODEL と同じ名前にする（セル9がその値を表示する）。
# gpu_memory_utilization は「そのvLLMが使ってよいGPU全体の割合」で、同居させる
# サービスの合計がGPU全体を超えないようにする。値は実機で未確認の初期値である。
#   chat 0.80  : L4の約22.5GiBのうち約18GiB。gpt-oss-20b(MXFP4)の重み・中間バッファ・
#                CUDAグラフを引いた残りがKVキャッシュになる。
#                実測（2026-11-23, L4）: 0.74ではKVキャッシュが0.37GiBしか残らず、
#                max_model_len 32768に要る0.85GiBに足りず起動に失敗した（0.74での
#                最大長の見積りは12032）。0.06増やすと約1.35GiB増え、約1.7GiBになる。
#   embed 0.10 : 約2.2GiB。bge-m3の重み約1.1GiB＋長文の中間バッファ。未検証（次の実行で確かめる）
#   合計0.90   : 残り約10%を、2プロセス分のCUDAコンテキストなどの固定費に残す
#   vlm 0.90   : 単独で動かす。Qwen3-VL-8Bの重み約17GiB＋KVキャッシュ。未検証
# 起動時に CUDA out of memory やKVキャッシュ不足で落ちたら、max_model_len を下げるか割合を調整する。
VLLM_SERVICES = {
    "chat": {
        "route": "chat",
        "port": 8001,
        "model": "openai/gpt-oss-20b",
        "served_name": "gpt-oss-20b",
        "gpu_memory_utilization": "0.80",
        "args": [
            "--max-model-len",
            "32768",
            # gpt-oss向けの公式の手順。道具呼び出し（Claude Code）に要る
            "--enable-auto-tool-choice",
            "--tool-call-parser",
            "openai",
        ],
    },
    "embed": {
        "route": "embed",
        "port": 8002,
        "model": "BAAI/bge-m3",
        "served_name": "bge-m3",
        "gpu_memory_utilization": "0.10",
        # bge-m3 は config.json が XLMRobertaModel と宣言しているため、公式文書どおり
        # hf-overrides で埋め込み用のモデルに差し替え、pooling の task を embed にする
        "args": [
            "--runner",
            "pooling",
            "--hf-overrides",
            '{"architectures": ["BgeM3EmbeddingModel"]}',
            "--pooler-config.task",
            "embed",
            "--max-model-len",
            "8192",
        ],
    },
    "vlm": {
        "route": "vlm",
        "port": 8003,
        "model": "Qwen/Qwen3-VL-8B-Instruct",
        "served_name": "qwen3-vl-8b",
        "gpu_memory_utilization": "0.90",
        # 画像だけを扱う。動画を許すと長い動画用のメモリを確保してしまう
        "args": ["--max-model-len", "16384", "--limit-mm-per-prompt", '{"video": 0}'],
    },
}
VLLM_PROFILES = {"chat+embed": ["chat", "embed"], "vlm": ["vlm"]}
# 初回はモデルのダウンロードとCUDAグラフの準備で10分以上かかることがある（1サービスあたり）
VLLM_START_TIMEOUT_SECONDS = 1800


def active_services() -> dict:
    """VLLM_PROFILE で選んだ組のサービス。実行時に読むので、値を変えても追従する。"""
    if VLLM_PROFILE not in VLLM_PROFILES:
        raise ValueError(
            f"VLLM_PROFILE は {sorted(VLLM_PROFILES)} のどれかにしてください（現在: {VLLM_PROFILE!r}）"
        )
    return {name: VLLM_SERVICES[name] for name in VLLM_PROFILES[VLLM_PROFILE]}


def vllm_command(service: dict) -> list:
    return [
        f"{VLLM_VENV}/bin/vllm",
        "serve",
        service["model"],
        # プロキシだけが届けばよいので、外へは公開しない
        "--host",
        "127.0.0.1",
        "--port",
        str(service["port"]),
        "--served-model-name",
        service["served_name"],
        "--gpu-memory-utilization",
        service["gpu_memory_utilization"],
        *service["args"],
    ]


def vllm_env() -> dict:
    """vLLMの仮想環境を有効化したのと同じ環境変数。

    vllm download-kernels は「同じPython環境で」実行する前提で、vllm serve も
    JITコンパイル用のツール（ninja等）を仮想環境の bin から探す。
    """
    return {
        **os.environ,
        "VIRTUAL_ENV": VLLM_VENV,
        "PATH": f"{VLLM_VENV}/bin:" + os.environ.get("PATH", ""),
    }


In [ ]:
# 1. GPU（L4）を確認する。
# セル0の値はL4のVRAM 24GBを前提に決めてある。T4（16GB）などが割り当てられると
# vLLMの起動途中で分かりにくい失敗をするため、ここで先に止める。
import subprocess

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_info = gpu_check.stdout.strip()
if gpu_check.returncode != 0 or not gpu_info:
    raise RuntimeError(
        "GPUが見えていません。ランタイムのタイプがGPU（L4）か確認してください: "
        + gpu_check.stderr
    )
if "L4" not in gpu_info:
    raise RuntimeError(
        f"L4以外のGPUが割り当てられています（{gpu_info}）。"
        "ランタイム → ランタイムのタイプを変更 でL4を選び直してください。"
    )
print("GPU確認OK:", gpu_info)

In [ ]:
# 2. vLLMを専用の仮想環境へ入れる（数分かかる）。
# Colab同梱の torch と vLLM が要求する torch のバージョンが食い違うため、
# Colab本体の環境には入れず VLLM_VENV に分ける。--allow-existing は再実行時に
# 既存の仮想環境を作り直さずに使い回すため。--seed は仮想環境に pip を入れるため
# （vLLM公式の uv 手順と同じ。download-kernels が追加パッケージを入れる際に要る）。
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "venv", VLLM_VENV, "--python", "3.12", "--seed", "--allow-existing"], check=True
)
subprocess.run(
    ["uv", "pip", "install", "--python", f"{VLLM_VENV}/bin/python", "vllm", "--torch-backend=auto"],
    check=True,
)
# FlashInferの事前コンパイル済みカーネルを取得し、起動時のコンパイル待ちを避ける
subprocess.run([f"{VLLM_VENV}/bin/vllm", "download-kernels"], check=True, env=vllm_env())
print("vLLMの導入OK")

In [ ]:
# 3. 選んだ組のvLLMをバックグラウンドで起動し、/v1/models が応答するまで待つ。
# 同居させる組は1つずつ順に起動する。同時に起動すると、お互いのメモリ確保が
# 重なって空き容量の確認を誤るため。起動途中で落ちた場合（VRAM不足など）は
# 待ち時間を使い切らず、すぐにログの末尾を表示して止める。ログ全体は vllm-<名前>.log にある。
import subprocess
import time
from pathlib import Path

import requests

# 再実行時、前回のvLLMがGPUメモリとポートを掴んだままだと新しいプロセスが起動に失敗する。
# 組を替えた場合に前の組が残らないよう、名前に関わらず全て止める。
for previous in globals().get("vllm_procs", {}).values():
    if previous.poll() is None:
        previous.terminate()
        previous.wait(timeout=60)

VLLM_COMMANDS = {name: vllm_command(service) for name, service in active_services().items()}
vllm_procs = {}
vllm_logs = {}


def show_vllm_log_tail(name):
    # V1エンジンでは本当の原因（CUDA out of memory 等）がエンジン側の子プロセスから
    # 先に出て、その後にAPIサーバー側の長いトレースバック（"See root cause above"）が
    # 続く。末尾だけでは原因が見えないため、エラーらしき行を先に抜き出して見せる。
    vllm_logs[name].flush()
    text = Path(f"vllm-{name}.log").read_text(encoding="utf-8", errors="replace")
    suspects = [
        line
        for line in text.splitlines()
        if "Error" in line or "out of memory" in line.lower()
    ]
    print(f"--- vllm-{name}.log のエラー行 ---")
    print("\n".join(suspects[-30:]) or "（該当なし）")
    print(f"--- vllm-{name}.log（末尾） ---")
    print(text[-3000:])


def wait_until_ready(name, port):
    proc = vllm_procs[name]
    for _ in range(VLLM_START_TIMEOUT_SECONDS // 5):
        if proc.poll() is not None:
            show_vllm_log_tail(name)
            raise RuntimeError(
                f"vLLM（{name}）が起動途中で終了しました（終了コード {proc.returncode}）。上のログを確認してください"
            )
        try:
            if requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=5).status_code == 200:
                print(f"vLLM（{name}）起動確認OK")
                return
        except requests.RequestException:
            pass
        time.sleep(5)
    show_vllm_log_tail(name)
    raise RuntimeError(f"vLLM（{name}）が{VLLM_START_TIMEOUT_SECONDS}秒以内に応答しませんでした")


for name, service in active_services().items():
    vllm_logs[name] = Path(f"vllm-{name}.log").open("w", encoding="utf-8")
    vllm_procs[name] = subprocess.Popen(
        VLLM_COMMANDS[name], stdout=vllm_logs[name], stderr=subprocess.STDOUT, env=vllm_env()
    )
    wait_until_ready(name, service["port"])


In [ ]:
# 4. プロキシ用のAPIキーを発行する。手元からは Authorization: Bearer <キー> として送る。
# RAGアプリ側は、このキーを VLLM_API_KEY に入れると同じ形式で送る（値はセル9が表示する）。
import os
import secrets

PROXY_API_KEY = secrets.token_urlsafe(24)
# プロキシは起動時にこの環境変数を読む。生成ファイルへキーを埋め込まない。
os.environ["PROXY_API_KEY"] = PROXY_API_KEY
print("APIキー（手元から試す際に Authorization: Bearer <キー> として送る）:")
print(PROXY_API_KEY)


In [ ]:
# 5. リバースプロキシ本体を書き出す。
# Authorization: Bearer ヘッダーを検証したうえで、パスの先頭（/chat・/embed・/vlm）で
# 転送先のvLLMを選び、その先頭を外して /v1/... をストリーミング転送する
# （/v1/chat/completions のSSEストリームもそのまま通す）。こうすると、ngrokのURLが
# 1つで済む。認証より先に振り分けを答えると、どのルートがあるかを未認証の相手に
# 教えてしまうため、401を先に返す。
# run_ollama_server.ipynb のプロキシを元にしているが、認証方式（Bearer）と
# 振り分けが違う。Colabからリポジトリを読まない構成のため写しにしている
# （技術的負債: 共通部分を直すときは両方のノートブックを直すこと）。
import json
from pathlib import Path

proxy_code = """
import hmac
import os

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = os.environ["PROXY_API_KEY"]
# パスの先頭 -> 転送先のvLLM
ROUTES = __ROUTES__

app = FastAPI()
_clients = {
    prefix: httpx.AsyncClient(
        base_url=base_url,
        timeout=httpx.Timeout(connect=10.0, read=600.0, write=60.0, pool=10.0),
    )
    for prefix, base_url in ROUTES.items()
}

_HOP_BY_HOP = {"host", "x-api-key", "authorization", "content-length", "connection", "transfer-encoding"}


def _is_authorized(request: Request) -> bool:
    supplied = request.headers.get("authorization", "")
    return hmac.compare_digest(supplied.encode(), f"Bearer {API_KEY}".encode())


@app.api_route("/{path:path}", methods=["GET", "POST"])
async def proxy(path: str, request: Request):
    if not _is_authorized(request):
        return JSONResponse({"error": "unauthorized"}, status_code=401)

    prefix, _, rest = path.partition("/")
    client = _clients.get(prefix)
    if client is None:
        return JSONResponse({"error": "unknown route"}, status_code=404)

    body = await request.body()
    upstream_req = client.build_request(
        request.method,
        f"/{rest}",
        params=request.query_params,
        content=body,
        headers=[(k, v) for k, v in request.headers.items() if k.lower() not in _HOP_BY_HOP],
    )
    try:
        upstream = await client.send(upstream_req, stream=True)
    except httpx.TimeoutException:
        return JSONResponse({"error": "vLLM upstream timed out"}, status_code=504)
    except httpx.RequestError:
        return JSONResponse({"error": "Unable to reach vLLM upstream"}, status_code=502)

    async def body_iter():
        try:
            async for chunk in upstream.aiter_raw():
                yield chunk
        finally:
            await upstream.aclose()

    return StreamingResponse(
        body_iter(),
        status_code=upstream.status_code,
        headers={k: v for k, v in upstream.headers.items() if k.lower() not in _HOP_BY_HOP},
    )
""".replace(
    "__ROUTES__",
    json.dumps(
        {s["route"]: f"http://127.0.0.1:{s['port']}" for s in active_services().values()}
    ),
)

Path("vllm_proxy.py").write_text(proxy_code, encoding="utf-8")
print("vllm_proxy.py を書き出しました（ルート:", ", ".join(s["route"] for s in active_services().values()) + "）")


In [ ]:
# 6. 認証プロキシに必要なパッケージを入れ、プロキシをバックグラウンドで起動する。
# プロキシはColab本体の環境で動かす（vLLMの仮想環境とは別。vLLMとはHTTPでしか話さない）。
# セル4(APIキー再発行)やこのセル自体を再実行すると、直前に起動したプロキシが
# 8000番を掴んだまま古いAPIキーで生き続け、新しいプロキシがバインドに失敗して
# 即終了する。再実行しても安全なように、起動前に前回のプロセスを必ず止める。
import subprocess
import time
from pathlib import Path

import requests

subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn[standard]", "httpx", "pyngrok"], check=True)

if "proxy_proc" in globals() and proxy_proc.poll() is None:
    proxy_proc.terminate()
    proxy_proc.wait(timeout=10)

proxy_log_path = Path("proxy.log")
proxy_log = proxy_log_path.open("w", encoding="utf-8")
proxy_proc = subprocess.Popen(
    ["uvicorn", "vllm_proxy:app", "--host", "0.0.0.0", "--port", str(PROXY_PORT)],
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
)

# 先頭のルートの /v1/models が通れば、認証と転送の両方が動いている
first_route = next(iter(active_services().values()))["route"]
for _ in range(30):
    try:
        r = requests.get(
            f"http://127.0.0.1:{PROXY_PORT}/{first_route}/v1/models",
            headers={"Authorization": f"Bearer {PROXY_API_KEY}"},
            timeout=2,
        )
        if r.status_code == 200:
            print("プロキシ起動確認OK（vLLMまで疎通）")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    proxy_log.flush()
    print("--- proxy.log ---")
    print(proxy_log_path.read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError("プロキシの起動に失敗しました")


In [ ]:
# 7. ngrokでプロキシ（8000番）を外部公開する。
# authtokenはngrokダッシュボード（https://dashboard.ngrok.com/get-started/your-authtoken）から取得する
from getpass import getpass

from google.colab import userdata
from pyngrok import conf, ngrok

# Colabのシークレット（鍵アイコン）に NGROK_AUTHTOKEN があれば使い、毎回の入力を省く。
# 未登録・アクセス未許可・取得のタイムアウトのときだけ入力に切り替える。それ以外の失敗は隠さない。
# タイムアウトは、シークレットがColabのUIから実行したときしか取得できないことと、
# アクセス許可の確認に応答しなかったときに起きる。
try:
    ngrok_token = (userdata.get("NGROK_AUTHTOKEN") or "").strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
    ngrok_token = ""

if not ngrok_token:
    ngrok_token = getpass("ngrok authtoken: ").strip()

if not ngrok_token:
    raise ValueError("ngrok authtoken が入力されていません。")

conf.get_default().auth_token = ngrok_token

# 既存のトンネルがあれば閉じてから張り直す（再実行時の重複防止）
ngrok.kill()
public_url = ngrok.connect(PROXY_PORT, "http").public_url

print("公開URL（手元から試す際に使う。RAGアプリ用の設定値はセル9が表示する）:")
print(public_url)

In [ ]:
# 8. 動作と速度を確かめる。プロキシ経由（ngrokの手前まで同じ経路）でストリーミング生成し、
# 初トークンまでの時間・全体の時間・トークン/秒と、思考（reasoning）と回答（content）が
# それぞれ届いたかを表示する。Ollama版と比べるときは同じ質問を両方に投げる。
# 組が "vlm" のときは、画像なしの文章でVLMの生成を確かめる。
import json
import time

import requests


def measure_chat(base_url: str, api_key: str, model: str, prompt: str, max_tokens: int = 1024) -> dict:
    started = time.perf_counter()
    response = requests.post(
        f"{base_url}/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": True,
            "stream_options": {"include_usage": True},
            "max_tokens": max_tokens,
        },
        stream=True,
        timeout=600,
    )
    response.raise_for_status()

    first_token_at = None
    reasoning_chars = 0
    content_parts = []
    completion_tokens = None
    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data: "):
            continue
        data = line[len("data: "):]
        if data == "[DONE]":
            break
        chunk = json.loads(data)
        if chunk.get("usage"):
            completion_tokens = chunk["usage"].get("completion_tokens")
        for choice in chunk.get("choices", []):
            delta = choice.get("delta", {})
            # vLLMのバージョンによって思考のフィールド名が reasoning / reasoning_content と異なる
            reasoning = delta.get("reasoning") or delta.get("reasoning_content") or ""
            content = delta.get("content") or ""
            if (reasoning or content) and first_token_at is None:
                first_token_at = time.perf_counter()
            reasoning_chars += len(reasoning)
            content_parts.append(content)

    total_seconds = time.perf_counter() - started
    return {
        "first_token_seconds": None if first_token_at is None else first_token_at - started,
        "total_seconds": total_seconds,
        "completion_tokens": completion_tokens,
        # 初トークンまでの待ち（プロンプト処理）を除いた生成速度。Ollamaの eval rate と同じ定義
        "decode_tokens_per_second": (
            None
            if completion_tokens is None or first_token_at is None
            else completion_tokens / (total_seconds - (first_token_at - started))
        ),
        "reasoning_chars": reasoning_chars,
        "content": "".join(content_parts),
    }


# 生成できるサービス（埋め込みは /v1/chat/completions を持たない）
generation_service = next(s for s in active_services().values() if s["route"] in ("chat", "vlm"))
speed_result = measure_chat(
    f"http://127.0.0.1:{PROXY_PORT}/{generation_service['route']}",
    PROXY_API_KEY,
    generation_service["served_name"],
    "日本の首都はどこですか？一文で答えてください。",
)
for key, value in speed_result.items():
    print(f"{key}: {value}")
if not speed_result["content"]:
    print(
        "回答（content）が空です。思考（reasoning）だけで max_tokens を使い切った可能性があります。"
        "max_tokens を増やして再実行してください。"
    )


In [ ]:
# 9. 開発機の .env に書く値を表示する。表示するだけで、ファイルには書かない。
# 末尾の / は取る（RAG側は /v1 を自分で足すため、URLを /v1 で終わらせてはいけない）。
# 組を替えたときは、その組のサービスの行だけが出る。
base_url = public_url.rstrip("/")
env_prefixes = {"chat": "VLLM_CHAT", "embed": "VLLM_EMBED", "vlm": "VLLM_VLM"}

print("開発機の .env に次の行を書く（組:", VLLM_PROFILE + "）:")
for service in active_services().values():
    prefix = env_prefixes[service["route"]]
    print(f"{prefix}_URL={base_url}/{service['route']}")
    print(f"{prefix}_MODEL={service['served_name']}")
print(f"VLLM_API_KEY={PROXY_API_KEY}")


## 手元から試す

セル7のURLとセル4のAPIキーを使い、手元の PowerShell から呼び出す。`Authorization: Bearer` が無い、または違うと401になる。URLの先頭のパス（`/chat`・`/embed`・`/vlm`）が転送先を決める。

```powershell
$url = "https://xxxx.ngrok-free.app"   # セル7で表示されたURL
$key = "セル4で表示されたAPIキー"
$headers = @{ Authorization = "Bearer $key" }

# モデル一覧（chat+embed の組）
Invoke-RestMethod -Uri "$url/chat/v1/models" -Headers $headers
Invoke-RestMethod -Uri "$url/embed/v1/models" -Headers $headers

# 生成（非ストリーム）。gpt-oss は思考を message.reasoning（版により reasoning_content）、回答を message.content に分けて返す
$body = @{
    model = "gpt-oss-20b"
    messages = @(@{ role = "user"; content = "日本の首都はどこですか？" })
    max_tokens = 1024
} | ConvertTo-Json -Depth 5
$res = Invoke-RestMethod -Uri "$url/chat/v1/chat/completions" -Method Post -Headers $headers `
    -ContentType "application/json; charset=utf-8" -Body ([Text.Encoding]::UTF8.GetBytes($body))
$res.choices[0].message.content

# 埋め込み。bge-m3 は1024次元で返す
$embedBody = @{ model = "bge-m3"; input = @("日本の首都は東京です。") } | ConvertTo-Json
$emb = Invoke-RestMethod -Uri "$url/embed/v1/embeddings" -Method Post -Headers $headers `
    -ContentType "application/json; charset=utf-8" -Body ([Text.Encoding]::UTF8.GetBytes($embedBody))
$emb.data[0].embedding.Count
```

`vlm` の組では、`$url/vlm/v1/chat/completions` に `model = "qwen3-vl-8b"` で同じように送る。

**RAGアプリについて**: セル9が表示した `VLLM_*` の行を開発機の `.env` に書き、RAGアプリを起動し直すと、このColabのvLLMを使う。回答生成と埋め込みは `chat+embed`、画像説明は `vlm` の組で、組を切り替えるたびに `VLLM_*_URL` のホスト部分（ngrokのURL）とAPIキーが変わる点に注意する。

## 後片付け

GPU使用量とログを確認し、使い終わったらランタイムを切断する（ランタイム → ランタイムを接続解除して削除）。

```python
!nvidia-smi
for log_path in sorted(Path(".").glob("vllm-*.log")):
    print(log_path, log_path.read_text(encoding="utf-8", errors="replace")[-3000:])
```
